<div style="background-color: #ffffff; color: #000000; padding: 30px;">
<img src="../media/images/kisz_logo.png" width="192" height="69" align="right" style="margin-right: 50px; margin-bottom: 50px;">
<h1>Time Series Analysis and Forecasting</h1>
</div>

<div style="background-color: #f6a800; color: #ffffff; padding: 10px;">
<h2>Part A: Foundations and Data Exploration</h2>
<h2>Notebook A02: Visualizing Time Series</h2>
</div>

Before fitting any model, you need to look at the data. Plots reveal things that summary statistics hide: trends that have been accelerating for decades, seasonal cycles that repeat every 12 months, sudden structural breaks, or a distribution that is much wider in summer than in winter. This notebook covers the main visualization techniques you will use throughout the course.

---

**Contents**

1. [Imports and Data Loading](#1.-Imports-and-Data-Loading)
2. [Line Plots](#2.-Line-Plots)
3. [Identifying Trends with Rolling Means](#3.-Identifying-Trends-with-Rolling-Means)
4. [Visualizing Seasonality](#4.-Visualizing-Seasonality)
5. [Polar Plots and Heatmaps](#5.-Polar-Plots-and-Heatmaps)
6. [Lag Plots](#6.-Lag-Plots)
7. [Autocorrelation and Partial Autocorrelation](#7.-Autocorrelation-and-Partial-Autocorrelation)

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="1.-Imports-and-Data-Loading">1. Imports and Data Loading</h3>
</div>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

import nb_config

We use the CDC regional air temperature dataset again. If you have not prepared it yet, run Notebook [F01b](./F01b_Preparing_CDC_dataset.ipynb) first.

Most plots in this notebook use the Brandenburg/Berlin series as a worked example. It is a single, clean, monthly series, which keeps the code readable. The same techniques apply to all other columns and to the OPS dataset.

In [ ]:
df = pd.read_parquet(nb_config.CDC_TEMP_PATH)

# Single-series shortcut used throughout the notebook
bb_ser = df["Brandenburg/Berlin"]
bb_ser.name = "Brandenburg/Berlin"

print(f"Series: {bb_ser.name}")
print(f"Range:  {bb_ser.index.min().date()} to {bb_ser.index.max().date()}")
print(f"Length: {len(bb_ser)} months")

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="2.-Line-Plots">2. Line Plots</h3>
</div>

The line plot is the starting point for any time series analysis. It connects observations in chronological order and lets you quickly spot the overall shape of the data: is there a long-term direction? Are there repeating patterns? Any obvious spikes or dips?

Start with the full series to get the big picture, then zoom in to a shorter window to see finer structure.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))

ax.plot(bb_ser, color="steelblue", linewidth=0.8, alpha=0.9)

ax.set_title("Monthly air temperature in Brandenburg/Berlin (full series)", fontsize=14, fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Temperature (°C)")
ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

With 140 years of data packed into one plot, the seasonal oscillations blur together. Zooming in to a shorter window makes the structure much clearer.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))

ax.plot(bb_ser["2010":], color="steelblue", linewidth=1.2)

ax.set_title("Monthly air temperature in Brandenburg/Berlin (2010 onwards)", fontsize=14, fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Temperature (°C)")
ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

You can also plot several series together to compare them. Keep the number of lines small enough that the plot stays readable. Five or six is usually the maximum before it becomes cluttered.

In [ ]:
regions = ["Deutschland", "Bayern", "Schleswig-Holstein", "Sachsen"]

fig, ax = plt.subplots(figsize=(14, 5))

for region in regions:
    ax.plot(df[region]["2010":], label=region.replace("_", " ").title(), linewidth=1.0, alpha=0.85)

ax.set_title("Monthly air temperature by region (2010 onwards)", fontsize=14, fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Temperature (°C)")
ax.legend(loc="upper left", framealpha=0.7)
ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

**Exercise.** Plot the full Deutschland series alongside its year-on-year difference (`bb_ser.diff(12)`). What does the difference series tell you that the raw series does not?

In [ ]:
# Your solution here


---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="3.-Identifying-Trends-with-Rolling-Means">3. Identifying Trends with Rolling Means</h3>
</div>

A trend is the long-term direction in the data. On a raw monthly series it is usually hidden behind seasonal variation and noise. A rolling mean smooths those out by replacing each point with the average of the surrounding window.

The window size controls how much smoothing you get. A short window (e.g. 12 months) follows the data closely and still shows seasonal bumps. A long window (e.g. 120 months) removes almost everything except the multi-decade drift.

Adding a confidence interval around the rolling mean gives a sense of how much the series varies within each window. We use the standard error of the mean (rolling std divided by the square root of the window size) to build a 95% band.

In [ ]:
windows = [36, 120, 240]  # 3 years, 10 years, 20 years

fig, axes = plt.subplots(len(windows), 1, figsize=(14, 5 * len(windows)), sharex=True)

for i, window in enumerate(windows):
    rolling_mean = bb_ser.rolling(window=window, min_periods=24).mean()
    rolling_std  = bb_ser.rolling(window=window, min_periods=24).std()
    se       = rolling_std / np.sqrt(window)
    ci_upper = rolling_mean + 1.96 * se
    ci_lower = rolling_mean - 1.96 * se

    axes[i].plot(bb_ser, color="gray", alpha=0.4, linewidth=0.7, label="Raw data")
    axes[i].plot(rolling_mean, color="steelblue", linewidth=2, label=f"{window}-month rolling mean")
    axes[i].fill_between(bb_ser.index, ci_lower, ci_upper, color="steelblue", alpha=0.15, label="95% CI")
    axes[i].set_title(f"Window: {window} months ({window // 12} years)", fontsize=13, fontweight="bold")
    axes[i].set_ylabel("Temperature (°C)")
    axes[i].legend(loc="upper left", framealpha=0.7)
    axes[i].grid(axis="y", linestyle="--", alpha=0.4)

axes[-1].set_xlabel("Date")
plt.tight_layout()
plt.show()

Notice how the 20-year rolling mean reveals a clear upward shift over the last few decades that is completely invisible on the raw series. This kind of long-term trend is important context when building a forecasting model: a model that ignores it will underforecast future values.

The confidence interval widens at the edges of the series where the rolling window has fewer observations to work with.

**Exercise.** Apply a 12-month rolling mean to all 17 regions and plot them on a single chart (no confidence interval needed). Which regions show the steepest upward trend since 1980?

In [ ]:
# Your solution here


---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="4.-Visualizing-Seasonality">4. Visualizing Seasonality</h3>
</div>

Seasonality is a pattern that repeats at a fixed, known period: every 12 months for yearly seasonality, every 7 days for weekly seasonality, and so on. The next three plots each look at the same seasonal structure from a different angle.

#### Seasonal line plot

A seasonal line plot overlays each year on the same monthly axis. When the lines track each other closely, the seasonal pattern is stable. When they diverge, something has changed (a trend, an anomaly, or a structural shift).

In [ ]:
# Month ordering used in all seasonal plots below
month_order = ["January", "February", "March", "April", "May", "June",
               "July", "August", "September", "October", "November", "December"]

fig, ax = plt.subplots(figsize=(14, 5))

# Use only recent decades to keep the colour palette readable
recent = bb_ser["2000":]
sns.lineplot(
    x=pd.Categorical(recent.index.month_name(), categories=month_order, ordered=True),
    y=recent,
    hue=recent.index.year,
    palette="RdBu_r",
    alpha=0.6,
    ax=ax,
)

ax.set_title("Seasonal line plot: Brandenburg/Berlin (2000 onwards)", fontsize=14, fontweight="bold")
ax.set_xlabel("Month")
ax.set_ylabel("Temperature (°C)")
ax.legend(title="Year", bbox_to_anchor=(1.01, 1), loc="upper left", framealpha=0.7)
ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

#### Box plot

A box plot summarises the distribution within each month across all years: the median, the interquartile range, and any outliers. It is a compact way to compare variability across seasons. Notice that summer months tend to have a tighter distribution than winter months.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

sns.boxplot(
    x=pd.Categorical(bb_ser.index.month_name(), categories=month_order, ordered=True),
    y=bb_ser,
    ax=ax,
)

ax.set_title("Monthly temperature distribution: Brandenburg/Berlin", fontsize=14, fontweight="bold")
ax.set_xlabel("Month")
ax.set_ylabel("Temperature (°C)")
ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

#### Violin plot

A violin plot adds a density estimate on top of the box plot, showing the full shape of the distribution for each month. This is useful when the distribution is not symmetric or has multiple modes, which a box plot would miss.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

sns.violinplot(
    x=pd.Categorical(bb_ser.index.month_name(), categories=month_order, ordered=True),
    y=bb_ser,
    ax=ax,
)

ax.set_title("Monthly temperature distribution: Brandenburg/Berlin (violin)", fontsize=14, fontweight="bold")
ax.set_xlabel("Month")
ax.set_ylabel("Temperature (°C)")
ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

**Exercise.** Build the same box plot for the `Sachsen` and `Schleswig-Holstein` series side by side on two subplots. Which region has more variable winters?

In [ ]:
# Your solution here


---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="5.-Polar-Plots-and-Heatmaps">5. Polar Plots and Heatmaps</h3>
</div>

Seasonal line plots work well but they use a linear x-axis, which breaks the circular nature of a yearly cycle: December and January are visually far apart even though they are adjacent in time. Polar plots fix that by wrapping the time axis into a circle.

#### Polar plot

Each spoke of the circle represents a month. The distance from the centre shows the temperature value. Each year is one closed loop.

One practical issue with seaborn's `lineplot` on a polar axis is that it treats December and January as separate years, so the loop for each year does not close. The fix is to append the January value of the next year to each year's trace so the line connects back to the start.

In [ ]:
# Use recent decades — plotting 140 years of loops is unreadable
polar_series = bb_ser["2000":]

# Append January of each following year so each annual loop closes
jan_values = polar_series[polar_series.index.month == 1].copy()
jan_values.index = jan_values.index - pd.DateOffset(years=1)  # shift back one year
extended = pd.concat([polar_series, jan_values]).sort_index()
extended = extended[~extended.index.duplicated(keep="first")]

# Angular positions: 12 months mapped to 0..2pi
theta = 2 * np.pi * (extended.index.month - 1) / 12
# Shift months that belong to a closing segment past December
is_closing_jan = (extended.index.month == 1) & (extended.index.shift(-1, freq="MS").month == 2)

fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(111, polar=True)

years = extended.index.year.unique()
palette = sns.color_palette("RdBu_r", len(years))

for j, yr in enumerate(years):
    mask = extended.index.year == yr
    yr_theta = theta[mask]
    yr_vals  = extended[mask].values
    # Close the loop by appending the first point
    yr_theta = np.append(yr_theta, yr_theta[0] + 2 * np.pi)
    yr_vals  = np.append(yr_vals,  yr_vals[0])
    ax.plot(yr_theta, yr_vals, color=palette[j], alpha=0.5, linewidth=1)

month_labels = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
ax.set_xticks(2 * np.pi * np.arange(12) / 12)
ax.set_xticklabels(month_labels)
ax.set_title("Monthly air temperature: Brandenburg/Berlin (2000 onwards)",
             va="bottom", fontsize=13, fontweight="bold", pad=20)

# Colour bar to indicate year
sm = plt.cm.ScalarMappable(cmap="RdBu_r", norm=plt.Normalize(years.min(), years.max()))
sm.set_array([])
plt.colorbar(sm, ax=ax, shrink=0.6, pad=0.1, label="Year")

plt.tight_layout()
plt.show()

#### Heatmap

A heatmap places years on one axis and months on the other, using colour to encode temperature. It is excellent for spotting anomalies (a single unusually cold February stands out as a single blue cell) and for seeing whether seasonal patterns are shifting over time.

In [ ]:
# Build a year x month pivot table
heatmap_df = bb_ser.to_frame(name="temperature")
heatmap_df["year"]  = bb_ser.index.year
heatmap_df["month"] = pd.Categorical(
    bb_ser.index.month_name(), categories=month_order, ordered=True
)

heatmap_data = heatmap_df.pivot_table(
    index="year", columns="month", values="temperature"
)

# Tick every 10 years to avoid overcrowding
year_ticks = heatmap_data.index[heatmap_data.index % 10 == 0]
tick_positions = [heatmap_data.index.get_loc(y) for y in year_ticks]

fig, ax = plt.subplots(figsize=(14, 10))

sns.heatmap(
    heatmap_data,
    cmap="RdBu_r",
    center=heatmap_data.mean().mean(),
    annot=False,
    linewidths=0,
    ax=ax,
    cbar_kws={"label": "Temperature (°C)"},
)

ax.set_yticks(tick_positions)
ax.set_yticklabels(year_ticks, rotation=0)
ax.set_title("Monthly air temperature heatmap: Brandenburg/Berlin", fontsize=14, fontweight="bold")
ax.set_xlabel("Month")
ax.set_ylabel("Year")

plt.tight_layout()
plt.show()

The shift from cooler to warmer colours in the bottom half of the chart (recent decades) relative to the top half (late 19th century) is the warming trend visible in the rolling mean plot, but now shown month by month.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="6.-Lag-Plots">6. Lag Plots</h3>
</div>

A lag plot is a scatter plot of a series against a shifted version of itself. On the x-axis you put the value at time t, and on the y-axis the value at time t + k, where k is the lag. If the points form a clear diagonal, the series is correlated with its own past at that lag.

Lag plots are a quick visual check before computing autocorrelation. They also reveal non-linear structure that a correlation coefficient would miss.

In [ ]:
lags_to_plot = [1, 6, 12, 24]

fig, axes = plt.subplots(1, len(lags_to_plot), figsize=(14, 4), sharey=True)

for ax, lag in zip(axes, lags_to_plot):
    ax.scatter(
        bb_ser,
        bb_ser.shift(lag),
        alpha=0.3,
        s=8,
        color="steelblue",
    )
    ax.set_title(f"Lag {lag}", fontsize=12, fontweight="bold")
    ax.set_xlabel("Value at t")
    if lag == lags_to_plot[0]:
        ax.set_ylabel("Value at t + lag")
    ax.axline((0, 0), slope=1, color="gray", linestyle="--", linewidth=0.8, alpha=0.6)
    ax.grid(linestyle="--", alpha=0.3)

fig.suptitle("Lag plots: Brandenburg/Berlin", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

Lag 1 shows a strong positive diagonal: consecutive months tend to have similar temperatures. Lag 6 shows a negative relationship: what is cold in winter tends to be warm six months later (summer), and vice versa. Lag 12 shows a strong positive diagonal again, confirming the 12-month seasonal cycle. Lag 24 shows the same structure, just slightly weaker.

**Exercise.** Build lag plots for lags 3, 6, 9, and 12. Does the relationship at lag 3 or lag 9 look linear or more curved? What does that tell you about the data?

In [ ]:
# Your solution here


---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="7.-Autocorrelation-and-Partial-Autocorrelation">7. Autocorrelation and Partial Autocorrelation</h3>
</div>

The ACF and PACF plots quantify what the lag plots show visually. They are also the standard tool for choosing the order of ARIMA models, which we cover in Part B.

**ACF (Autocorrelation Function):** measures the correlation between the series and a lagged version of itself, for each lag from 0 to some maximum. The shaded band shows the 95% confidence interval under the assumption of no autocorrelation. Spikes that go beyond the band are statistically significant.

**PACF (Partial Autocorrelation Function):** measures the same thing, but removes the contribution of all shorter lags first. If lag 12 is significant in the PACF after controlling for lags 1 through 11, that is strong evidence of direct 12-month dependence rather than it being a side effect of shorter-lag correlations.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))

plot_acf(bb_ser, lags=48, ax=ax, color="steelblue", vlines_kwargs={"colors": "steelblue"})

ax.set_title("ACF: Brandenburg/Berlin (48 lags)", fontsize=14, fontweight="bold")
ax.set_xlabel("Lag (months)")
ax.set_ylabel("Autocorrelation")
ax.set_ylim(-1.1, 1.1)
ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

The ACF shows large positive spikes at lags 12, 24, and 36, and large negative spikes at lags 6, 18, and 30. This is the textbook signature of yearly seasonality on monthly data: the series is most similar to itself 12 months ago, and most dissimilar to itself 6 months ago.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))

plot_pacf(bb_ser, lags=48, ax=ax, color="steelblue", vlines_kwargs={"colors": "steelblue"})

ax.set_title("PACF: Brandenburg/Berlin (48 lags)", fontsize=14, fontweight="bold")
ax.set_xlabel("Lag (months)")
ax.set_ylabel("Partial autocorrelation")
ax.set_ylim(-1.1, 1.1)
ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

The PACF shows a very large spike at lag 1, then a sharp drop. This suggests a strong direct one-step dependence: knowing last month's temperature is the most useful single predictor of this month's temperature. The spike at lag 12 is also significant after controlling for all shorter lags, confirming that last year's same-month value carries independent predictive value beyond what shorter lags already capture.

**Exercise.** Compute and plot the ACF for the year-on-year differenced series (`bb_ser.diff(12).dropna()`). How does it compare to the ACF of the raw series? What has differencing removed?

In [ ]:
# Your solution here


---

You now have a toolkit for visual exploration of time series data. In the next notebook we look at missing data: how to detect it, understand why it is missing, and decide how to handle it before feeding the data into a model.